NN EXTRA

1. Introducción y Configuración Inicial

In [1]:
# nn_extra_solve_by_levi.ipynb
# ----------------------------------------------------
# Cuaderno didáctico para la resolución de ejercicios extra de Redes Neuronales

import tensorflow as tf
from tensorflow.keras import layers, models, datasets
import matplotlib.pyplot as plt
import numpy as np

# Configuración para reproducibilidad
tf.random.set_seed(42)

# Cargamos un dataset de ejemplo (CIFAR-10: 10 clases de imágenes de 32x32)
(x_train, y_train), (x_test, y_test) = datasets.cifar10.load_data()

# Normalizamos los píxeles al rango [0, 1]
x_train, x_test = x_train / 255.0, x_test / 255.0

print(f"Dataset cargado. Entrenamiento: {x_train.shape}, Test: {x_test.shape}")

 10870784/170498071 ━━━━━━━━━━━━━━━━━━━━ 7:01:48 159us/step

KeyboardInterrupt: 

2. Ejercicio 1: Capas Dropout

In [ ]:
### --- MODELO CON DROPOUT ---

model_dropout = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),
    
    # Añadimos Dropout del 25% tras la capa de pooling
    layers.Dropout(0.25), [cite: 100]
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25), [cite: 100]
    
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    
    # Dropout del 50% antes de la capa de clasificación final
    layers.Dropout(0.5), [cite: 100]
    layers.Dense(10, activation='softmax')
])

model_dropout.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])

print("Entrenando modelo con Dropout...")
history_dropout = model_dropout.fit(x_train, y_train, epochs=10, 
                                    validation_data=(x_test, y_test), batch_size=64)

3. Ejercicio 2: Capas BatchNormalization

In [ ]:
### --- MODELO CON BATCH NORMALIZATION ---

model_bn = models.Sequential([
    # Se suele aplicar justo antes o después de la función de activación
    layers.Conv2D(32, (3, 3), padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(), [cite: 101]
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(), [cite: 101]
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    layers.Dense(64),
    layers.BatchNormalization(), [cite: 101]
    layers.Activation('relu'),
    layers.Dense(10, activation='softmax')
])

model_bn.compile(optimizer='adam',
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])

print("Entrenando modelo con Batch Normalization...")
history_bn = model_bn.fit(x_train, y_train, epochs=10, 
                          validation_data=(x_test, y_test), batch_size=64)

4. Ejercicio 3: Data Augmentation con capas de Keras

In [ ]:
### --- DEFINICIÓN DE CAPAS DE AUGMENTATION ---

data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),        # Giro horizontal aleatorio [cite: 103, 104]
    layers.RandomRotation(0.1),             # Rotación aleatoria del 10% [cite: 103, 104]
    layers.RandomZoom(0.1),                 # Zoom aleatorio [cite: 103, 104]
])

### --- MODELO CON DATA AUGMENTATION INTEGRADO ---

model_aug = models.Sequential([
    # Pasamos las imágenes primero por la etapa de aumento de datos
    layers.Input(shape=(32, 32, 3)),
    data_augmentation, [cite: 102]
    
    # Resto de la red convolucional estándar
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model_aug.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

print("Entrenando modelo con Data Augmentation...")
history_aug = model_aug.fit(x_train, y_train, epochs=10, 
                            validation_data=(x_test, y_test), batch_size=64)

5. Ejercicio 4: Transfer Learning con una red ligera

In [ ]:
### --- TRANSFER LEARNING CON MOBILENETV2 ---

# Nota: MobileNetV2 espera imágenes un poco más grandes, pero Keras permite adaptarlo.
# Cargamos la red base sin la capa final de clasificación (include_top=False)
base_model = tf.keras.applications.MobileNetV2(input_shape=(32, 32, 3),
                                               include_top=False, [cite: 107]
                                               weights='imagenet')

# ¡CRUCIAL! Congelamos el modelo base para no alterar sus pesos entrenados
base_model.trainable = False [cite: 106]

# Construimos el modelo final
model_tl = models.Sequential([
    base_model, [cite: 105, 106]
    layers.GlobalAveragePooling2D(), # Reduce las dimensiones espaciales
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax') # Ajustado a nuestras 10 clases
])

model_tl.compile(optimizer='adam',
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])

print("Entrenando modelo con Transfer Learning (MobileNetV2)...")
history_tl = model_tl.fit(x_train, y_train, epochs=10, 
                          validation_data=(x_test, y_test), batch_size=64)

6. Comparativa Final de Rendimiento

In [ ]:
# Gráfica comparativa
plt.figure(figsize=(12, 6))

plt.plot(history_dropout.history['val_accuracy'], label='Con Dropout', linestyle='--')
plt.plot(history_bn.history['val_accuracy'], label='Con Batch Normalization', linestyle='-')
plt.plot(history_aug.history['val_accuracy'], label='Con Data Augmentation', linestyle='-.')
plt.plot(history_tl.history['val_accuracy'], label='Transfer Learning (MobileNetV2)', linestyle=':')

plt.title('Comparativa de Rendimiento en el Set de Validación')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()